# Tilawah — GPU transcription server

Runs the Quran-tuned Whisper model on this notebook's GPU and exposes it to your PC through a free Cloudflare quick tunnel.

**Colab:** `Runtime → Change runtime type → T4 GPU`, then run the cell below.

**Kaggle:** in Notebook settings enable `GPU T4` and `Internet`, then run the cell below.

When it prints `PUBLIC URL` and `TOKEN`, paste the three lines it shows into `backend\.env` on your PC, restart the Tilawah backend, and practice. **Keep the cell running** — closing it stops the server.

In [ ]:
%pip -q install faster-whisper fastapi uvicorn python-multipart requests

import os, re, secrets, socket, subprocess, threading, time, urllib.request
import requests

TOKEN = secrets.token_urlsafe(16)
os.environ['TILAWAH_TOKEN'] = TOKEN

# Kill leftovers from any previous run (interrupting a cell does NOT stop
# its subprocesses), then pick a fresh free port so a collision is impossible.
subprocess.run(['pkill', '-9', '-f', 'gpu_server.py'], check=False)
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], check=False)
time.sleep(2)

s = socket.socket()
s.bind(('', 0))
PORT = s.getsockname()[1]
s.close()
os.environ['TILAWAH_GPU_PORT'] = str(PORT)
print(f'Using port {PORT}')

# ── the server (same code as cloud/gpu_server.py in the repo) ──
server_code = r'''
import io, os, secrets
import uvicorn
from fastapi import FastAPI, Header, HTTPException, UploadFile
from faster_whisper import WhisperModel

MODEL_ID = os.environ.get('TILAWAH_GPU_MODEL', 'OdyAsh/faster-whisper-base-ar-quran')
TOKEN = os.environ.get('TILAWAH_TOKEN') or secrets.token_urlsafe(16)
PORT = int(os.environ.get('TILAWAH_GPU_PORT', '9000'))

def pick_device():
    # fp16 needs tensor-core GPUs (T4 and newer); older cards like the
    # P100 fall back to the best type they support (float32 is still fast).
    try:
        import ctranslate2
        if ctranslate2.get_cuda_device_count() > 0:
            supported = ctranslate2.get_supported_compute_types('cuda')
            for ct in ('float16', 'int8_float16', 'int8_float32', 'float32'):
                if ct in supported:
                    return 'cuda', ct, 5
    except Exception:
        pass
    return 'cpu', 'int8', 1

DEVICE, COMPUTE, BEAM = pick_device()
print(f'Loading {MODEL_ID} on {DEVICE} ({COMPUTE}, beam={BEAM}) ...', flush=True)
model = WhisperModel(MODEL_ID, device=DEVICE, compute_type=COMPUTE)
import numpy as np
list(model.transcribe(np.zeros(8000, dtype=np.float32), language='ar', beam_size=1)[0])
print('Model ready.', flush=True)

app = FastAPI(title='Tilawah GPU server')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID, 'device': DEVICE, 'compute': COMPUTE, 'beamSize': BEAM}

@app.post('/transcribe')
async def transcribe(audio: UploadFile, x_tilawah_token: str = Header(default='')):
    if x_tilawah_token != TOKEN:
        raise HTTPException(401, 'Bad or missing X-Tilawah-Token header')
    data = await audio.read()
    if not data:
        raise HTTPException(422, 'Empty audio upload')
    segments, _ = model.transcribe(io.BytesIO(data), language='ar', beam_size=BEAM,
                                   vad_filter=False, condition_on_previous_text=False,
                                   no_repeat_ngram_size=3, repetition_penalty=1.05)
    return {'transcription': ' '.join(s.text.strip() for s in segments).strip()}

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='warning')
'''
open('gpu_server.py', 'w').write(server_code)

server = subprocess.Popen(['python', 'gpu_server.py'], env={**os.environ},
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
threading.Thread(target=lambda: [print('[server]', l, end='') for l in server.stdout], daemon=True).start()

print('Waiting for the model to load (first run downloads ~150 MB)...')
for _ in range(300):
    if server.poll() is not None:
        raise RuntimeError('The server crashed — scroll up to the [server] lines for the error.')
    try:
        requests.get(f'http://127.0.0.1:{PORT}/health', timeout=2)
        break
    except Exception:
        time.sleep(2)
print('Server is up. Opening the tunnel...')

urllib.request.urlretrieve(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', 'cloudflared')
os.chmod('cloudflared', 0o755)
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

print('=' * 64)
print('PUBLIC URL :', url)
print('TOKEN      :', TOKEN)
print()
print(r'On your PC, paste these three lines into  backend\.env')
print('(overwrite any old URL/TOKEN), then restart the Tilawah backend:')
print()
print('  TILAWAH_ASR_BACKEND=remote')
print(f'  TILAWAH_ASR_REMOTE_URL={url}')
print(f'  TILAWAH_ASR_REMOTE_TOKEN={TOKEN}')
print('=' * 64)
print('Keep this cell running while you practice.')
tun.wait()